# 00: Colab CPU Replay

This notebook runs on **CPU** and uses the DeepSeek API for:
- Generating debate traces
- Counterfactual replay
- Building datasets

**Persistence**: All data is stored on HF Hub.

## Setup

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/Arshia-HZ/causal-mas-distill.git
%cd causal-mas-distill
!pip install -r requirements-api.txt

In [ ]:
# HF Hub setup
from huggingface_hub import create_repo, snapshot_download

create_repo("Arshia-HZ/causal-mas-distill-data", repo_type="dataset", exist_ok=True)
try:
    snapshot_download("Arshia-HZ/causal-mas-distill-data", repo_type="dataset", local_dir="data")
except Exception as e:
    print("Empty repo, starting fresh:", e)

In [ ]:
# Set API key from Colab Secrets
import os
from google.colab import userdata
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")

## Step 1: Select Hard Problems

In [ ]:
# Run problem selection
!python scripts/00_select_hard_problems.py \
    --input data/problems.json \
    --output data/hard_problems.json \
    --difficulty-threshold 0.5 \
    --max-problems 1000

## Step 2: Generate Debates

In [ ]:
!python scripts/01_generate_debates.py \
    --problems data/hard_problems.json \
    --output data/traces/debates.json \
    --api-url https://api.deepseek.com/v1 \
    --model deepseek-v4-flash \
    --max-rounds 3

## Step 3: Counterfactual Replay

In [ ]:
!python scripts/02_counterfactual_replay.py \
    --traces data/traces/debates.json \
    --output data/counterfactuals/results.json \
    --api-url https://api.deepseek.com/v1 \
    --model deepseek-v4-flash \
    --sample-size 100

## Step 4: Noise Floor Estimation

In [ ]:
!python scripts/02b_noise_floor.py \
    --traces data/traces/debates.json \
    --counterfactuals data/counterfactuals/results.json \
    --output data/noise_floor/results.json \
    --api-url https://api.deepseek.com/v1 \
    --model deepseek-v4-flash

## Step 5: Build Datasets

In [ ]:
!python scripts/03_build_datasets.py \
    --traces data/traces/debates.json \
    --utilities data/counterfactuals/results.json \
    --output-dir data/datasets \
    --token-budget 100000

## Upload to HF Hub

In [ ]:
# Upload all data to HF Hub
from huggingface_hub import upload_folder
upload_folder(
    folder_path="data",
    repo_id="Arshia-HZ/causal-mas-distill-data",
    repo_type="dataset"
)